# 06 – Temperatura de Chama Adiabática

Quando um combustível queima em uma câmara isolada a pressão constante, toda a
energia química liberada é utilizada para aquecer os produtos. A temperatura
resultante é a **temperatura de chama adiabática** $T_\mathrm{ad}$ — um
parâmetro fundamental para o projeto de câmaras de combustão e turbinas.

Como a entalpia padronizada do `pyglenn` já contém a entalpia de formação de
cada espécie, o balanço de energia adiabático e isobárico é simplesmente a
*conservação da entalpia total*:

$$\sum_{\text{reagentes}} n_i\,H^\circ_i(T_\mathrm{entrada})
  \;=\; \sum_{\text{produtos}} n_j\,H^\circ_j(T_\mathrm{ad}).$$

Resolvemos esta equação não linear para $T_\mathrm{ad}$. O ar é modelado como
$1\ \mathrm{O_2} + 3{,}76\ \mathrm{N_2}$.

> **Nota sobre o estado padrão de gás ideal:** As propriedades dos polinômios
> NASA representam o **estado padrão de gás ideal** (1 bar). Em altas pressões
> ou para misturas com fases condensadas, correções de não idealidade (ex.:
> coeficientes de fugacidade, equações de estado) devem ser aplicadas — os
> valores brutos de `h_relative` do `pyglenn` são o limite de gás ideal.

In [ ]:
from pyglenn import ThermochemicalCalculator, R

_INDEX = {}

def species_id(calc, name):
    """Retorna o id no banco de dados da espécie cujo *nome* corresponde exatamente.

    ``get_available_species`` faz correspondência de substrings tanto no nome
    quanto na fórmula e limita o resultado a 20 linhas, de modo que nomes
    curtos como ``"O2"`` podem ser sufocados por entradas como ``"AL2O2"`` ou
    ``"CO2"``. Para sermos robustos, construímos um índice completo nome -> id
    uma única vez (armazenado em cache durante a sessão) e buscamos o nome
    exato nele.
    """
    if not _INDEX:
        _INDEX.update({s["name"]: s["id"] for s in calc.get_available_species("")})
    if name not in _INDEX:
        raise ValueError(f"Espécie {name!r} não encontrada no banco de dados")
    return _INDEX[name]

print("Constante universal dos gases R =", R, "J/(mol.K)")


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

from scipy.optimize import brentq

## 1. Entalpia da mistura e o balanço de energia

`mixture_enthalpy` soma $n_i H^\circ_i(T)$ sobre um dicionário
`{espécie: mols}`. `adiabatic_flame_T` encontra a temperatura dos produtos na
qual a entalpia dos produtos iguala a entalpia dos reagentes, usando um método
robusto de busca de raiz com intervalo (`brentq`). A entalpia dos produtos
cresce monotonicamente com $T$, portanto a raiz é única.

In [ ]:
def mixture_enthalpy(calc, counts, T):
    """Entalpia padronizada total de uma mistura {espécie: mols}, em J."""
    return sum(n * calc.calculate_properties(species_id(calc, name), T)["h_relative"]
               for name, n in counts.items())

def adiabatic_flame_T(calc, reactants, products, T_in=298.15, T_hi=6000.0):
    """Temperatura de chama adiabática a pressão constante, em K."""
    H_react = mixture_enthalpy(calc, reactants, T_in)
    f = lambda T: mixture_enthalpy(calc, products, T) - H_react
    return brentq(f, T_in, T_hi)

## 2. Metano/ar estequiométrico

$$\mathrm{CH_4} + 2(\mathrm{O_2} + 3{,}76\,\mathrm{N_2})
   \rightarrow \mathrm{CO_2} + 2\,\mathrm{H_2O} + 7{,}52\,\mathrm{N_2}.$$

O nitrogênio é inerte, mas carrega consigo uma grande parcela da energia
liberada.

In [ ]:
reactants = {"CH4": 1.0, "O2": 2.0, "N2": 2 * 3.76}
products  = {"CO2": 1.0, "H2O": 2.0, "N2": 2 * 3.76}

with ThermochemicalCalculator() as calc:
    Tad = adiabatic_flame_T(calc, reactants, products, T_in=298.15)
print(f"Temperatura de chama adiabática (CH4/ar, estequiométrico): {Tad:.1f} K "
      f"= {Tad-273.15:.0f} °C")
print("Referência sem dissociação: ~2320 K.")

## 3. Visualizando o balanço

Plotando a entalpia dos produtos em função da temperatura e traçando a entalpia
dos reagentes (constante) como uma linha horizontal, a interseção é
$T_\mathrm{ad}$.

In [ ]:
Tgrid = np.linspace(298.15, 3200, 80)
with ThermochemicalCalculator() as calc:
    H_prod = np.array([mixture_enthalpy(calc, products, T) for T in Tgrid]) / 1000.0
    H_react = mixture_enthalpy(calc, reactants, 298.15) / 1000.0

fig, ax = plt.subplots()
ax.plot(Tgrid, H_prod, label="produtos $H(T)$")
ax.axhline(H_react, color="C1", ls="--", label=r"reagentes $H(T_\mathrm{entrada})$")
ax.axvline(Tad, color="0.5", ls=":")
ax.scatter([Tad], [H_react], color="red", zorder=5, label=rf"$T_\mathrm{{ad}}$ = {Tad:.0f} K")
ax.set_xlabel("Temperatura [K]")
ax.set_ylabel("Entalpia da mistura [kJ]")
ax.set_title("Temperatura de chama adiabática como balanço de entalpia")
ax.legend()
plt.show()

## 4. Efeito da razão de equivalência

A **razão de equivalência** $\phi$ é a razão combustível/ar real dividida pela
estequiométrica. Misturas pobres ($\phi<1$) contêm excesso de ar que absorve
calor sem liberar mais energia, reduzindo $T_\mathrm{ad}$. Assumindo combustão
completa, para $\phi \le 1$ os produtos são CO₂, H₂O, o O₂ excedente e todo o
N₂.

In [ ]:
def flame_T_phi(calc, phi, T_in=298.15):
    """Temperatura de chama adiabática CH4/ar na razão de equivalência phi (<=1)."""
    # Fixa 1 mol de combustível; O2 estequiométrico = 2. Pobre => mais ar => O2_fornecido = 2/phi.
    o2 = 2.0 / phi
    n2 = o2 * 3.76
    reac = {"CH4": 1.0, "O2": o2, "N2": n2}
    prod = {"CO2": 1.0, "H2O": 2.0, "O2": o2 - 2.0, "N2": n2}
    prod = {k: v for k, v in prod.items() if v > 0}
    return adiabatic_flame_T(calc, reac, prod, T_in)

phis = np.linspace(0.5, 1.0, 26)
with ThermochemicalCalculator() as calc:
    Tad_phi = [flame_T_phi(calc, p) for p in phis]

fig, ax = plt.subplots()
ax.plot(phis, Tad_phi)
ax.set_xlabel(r"razão de equivalência $\phi$")
ax.set_ylabel(r"$T_\mathrm{ad}$ [K]")
ax.set_title("Misturas metano/ar mais pobres queimam mais frias")
plt.show()
print(f"estequiométrica (phi=1,0): {Tad_phi[-1]:.0f} K")
print(f"pobre           (phi=0,5): {Tad_phi[0]:.0f} K")

## 5. Efeito do pré-aquecimento dos reagentes

Pré-aquecer o ar de entrada (recuperação de calor, comum em turbinas a gás)
eleva a entalpia dos reagentes e, portanto, $T_\mathrm{ad}$. Aqui varremos a
temperatura de entrada para a mistura estequiométrica.

In [ ]:
T_in_grid = np.linspace(298.15, 800, 25)
with ThermochemicalCalculator() as calc:
    Tad_preheat = [adiabatic_flame_T(calc, reactants, products, T_in=Ti)
                   for Ti in T_in_grid]

fig, ax = plt.subplots()
ax.plot(T_in_grid, Tad_preheat)
ax.set_xlabel(r"temperatura de entrada $T_\mathrm{entrada}$ [K]")
ax.set_ylabel(r"$T_\mathrm{ad}$ [K]")
ax.set_title("Pré-aquecer os reagentes eleva a temperatura de chama")
plt.show()

## 6. Comparando combustíveis

Temperaturas de chama estequiométricas no ar (entrada a 298,15 K) para quatro
combustíveis. Todos usam sua própria demanda de oxigênio; hidrogênio e
hidrocarbonetos agrupam-se uma vez diluídos pelo nitrogênio atmosférico.

In [ ]:
FUEL_STOICH = {
    "H2":     ({"O2": 0.5}, {"CO2": 0, "H2O": 1}),
    "CH4":    ({"O2": 2.0}, {"CO2": 1, "H2O": 2}),
    "C3H8":   ({"O2": 5.0}, {"CO2": 3, "H2O": 4}),
    "C2H5OH": ({"O2": 3.0}, {"CO2": 2, "H2O": 3}),
}

rows = []
with ThermochemicalCalculator() as calc:
    for fuel, (o2d, prod) in FUEL_STOICH.items():
        nO2 = o2d["O2"]
        nN2 = nO2 * 3.76
        reac = {fuel: 1.0, "O2": nO2, "N2": nN2}
        products_f = {k: v for k, v in {**prod, "N2": nN2}.items() if v > 0}
        Tad_f = adiabatic_flame_T(calc, reac, products_f, T_in=298.15)
        rows.append({"combustível": fuel, "T_ad [K]": Tad_f})

fuel_df = pd.DataFrame(rows).set_index("combustível")
print(fuel_df.to_string())

fig, ax = plt.subplots()
ax.bar(fuel_df.index, fuel_df["T_ad [K]"], color="#c0392b")
ax.set_ylabel(r"$T_\mathrm{ad}$ [K]")
ax.set_ylim(2000, 2500)
ax.set_title("Temperatura de chama estequiométrica no ar")
plt.show()

## Ressalva: dissociação

Este modelo assume **combustão completa** para CO₂ e H₂O. Chamas reais acima de
~2000 K dissociam parcialmente os produtos (em CO, OH, H, O, NO, ...), o que
absorve energia e limita a temperatura de chama real **100–300 K abaixo** dessas
estimativas. O notebook 08 quantifica a dissociação via equilíbrio químico.

**A seguir:** o notebook 07 aplica essas ferramentas de combustão para comparar
biocombustíveis.